In [ ]:
import numpy as np
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt

In [ ]:
from src.utils import degree_3_polynomial
A = 1 
B = 2 
C = -0.5 
D = 0.04 
x_true = np.linspace(0, 10, 1000)
y_true = degree_3_polynomial(x_true, coeffs=[A, B, C, D], noise_std=0.8)

plt.plot(x_true, y_true, color="black")
plt.xlabel('X')
plt.ylabel('Y')

In [ ]:
sample_size = 20
sampling_indices = np.random.choice(len(x_true), size=sample_size, replace=False)
x_sample = x_true[sampling_indices]
y_sample = y_true[sampling_indices] 

plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--', alpha=0.5)
plt.scatter(x_sample, y_sample, color="red", label="Sampled Data")
plt.legend()


In [ ]:
from src.utils import define_model, train_multiple_times
model_complex = define_model(num_neurons=64, num_depth=1)

In [ ]:
x_test = np.linspace(-5, 15, 1000)
y_test = degree_3_polynomial(x_test, coeffs=[A, B, C, D], noise_std=0)
all_predictions = train_multiple_times(model_complex, x_sample, y_sample, x_test, n_runs=50)


In [ ]:
# Plot all predictions to show the variability due to different random initializations
plt.plot(x_test, y_test, color="black", label="True Function", linestyle='--')
for i in range(all_predictions.shape[0]):
    plt.plot(x_test, all_predictions[i], color="red", alpha=0.1)
plt.xlabel('X')
plt.ylabel('Y')
plt.ylim(-5, 30)
plt.xlim(-5, 15)
plt.scatter(x_sample, y_sample, color="blue", label="Sampled Data")

In [ ]:
from ipywidgets import interact, widgets

def plot_distribution_at_x(x_test, y_test, x_sample, y_sample, all_predictions, x_point):
    import seaborn as sns
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].plot(x_test, y_test, color="black", label="True Function", linestyle='--')
    ax[0].scatter(x_sample, y_sample, color="blue", label="Sampled Data")
    ax[0].set_ylim(-5, 30)
    ax[0].set_xlim(-5, 15)
    for i in range(all_predictions.shape[0]):
        ax[0].plot(x_test, all_predictions[i], color="red", alpha=0.1)
    # Highlight the predictions at the specific x_point
    ax[0].axvline(x=x_point, color="green", linestyle='--', label=f"x = {x_point:.1f}")
    # Plot the distribution of predictions at the specific x_point
    predictions_at_point = all_predictions[:, np.argmin(np.abs(x_test - x_point))]
    sns.histplot(predictions_at_point, kde=True, ax=ax[1], color="red")
    y_true_at_point = y_test[np.argmin(np.abs(x_test - x_point))]
    ax[1].axvline(x=y_true_at_point, color="black", linestyle='--', label=f"True y at x={x_point:.1f}")
    ax[1].set_xlabel("Predicted y values")
    ax[1].set_title(f"Distribution of Predictions at x={x_point:.1f}")
    #ax[1].set_xlim(-5, 30)
    ax[1].legend()

    fig.tight_layout()

x_point_slider = widgets.FloatSlider(value=5, min=-5, max=15, step=0.1, description='x_point')
interact(plot_distribution_at_x, \
            x_test=widgets.fixed(x_test), \
            y_test=widgets.fixed(y_test), \
            x_sample=widgets.fixed(x_sample), \
            y_sample=widgets.fixed(y_sample), \
            all_predictions=widgets.fixed(all_predictions), \
            x_point=x_point_slider)


Why does the exact same data give rise to many different functions? How does model complexity affect this uncertainty? 

In [ ]:
from tqdm import tqdm
model_size = [(3,1), (16,1), (64,1), (256,1), (3,2), (16,2), (64,2), (256,2)]
all_predictions_by_size = {}
for size in tqdm(model_size):
    model = define_model(num_neurons=size[0], num_depth=size[1])
    all_predictions_by_size[size] = train_multiple_times(model, x_sample, y_sample, x_test, n_runs=50)

In [ ]:
def plot_predictions_by_model_size(x_test, y_test, x_sample, y_sample, all_predictions_by_size, model_size):
    import seaborn as sns
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for i, size in enumerate(model_size):
        ax = axes[i//4, i%4]
        ax.plot(x_test, y_test, color="black", label="True Function", linestyle='--')
        ax.scatter(x_sample, y_sample, color="blue", label="Sampled Data")
        ax.set_ylim(-5, 30)
        ax.set_xlim(-5, 15)
        for j in range(all_predictions_by_size[size].shape[0]):
            ax.plot(x_test, all_predictions_by_size[size][j], color="red", alpha=0.1)
        ax.set_title(f"Model Size: {size[0]} neurons, {size[1]} layers")
    #axes[0].legend()
    fig.tight_layout()



In [ ]:
plot_predictions_by_model_size(x_test, y_test, x_sample, y_sample, all_predictions_by_size, model_size)

What counts as a "good" model of these? 

Why does the same data give rise to many different functions?

Check loss landscape by plotting the loss as a function of the model parameters. This can help visualize how different parameter values can lead to similar loss values, indicating that multiple functions can fit the same data well.